In [0]:
#spark
import pandas as pd
riders = pd.DataFrame({

    "rider_id": [1, 2, 3, 4],
    "registration_date": pd.to_datetime([
        "2023-01-01", "2023-01-05", "2023-01-10", "2023-01-15"
    ])
})

riders


In [0]:
customer_orders = pd.DataFrame({
    "order_id": [1,2,3,3,4,4,5,6,7,8,9,10,10],
    "customer_id": [201,201,202,202,203,203,204,201,205,202,203,204,204],
    "pizza_id": [1,1,1,2,1,2,1,2,2,1,1,1,1],
    "exclusions": ["","", "", "", "4", "4", None, None, None, None, "4", None, "2, 6"],
    "extras": ["","", "", None, "", "", "1", None, "1", None, "1, 5", None, "1, 4"],
    "order_time": pd.to_datetime([
        "2023-01-01 18:05:02",
        "2023-01-01 19:00:52",
        "2023-01-02 23:51:23",
        "2023-01-02 23:51:23",
        "2023-01-04 13:23:46",
        "2023-01-04 13:23:46",
        "2023-01-08 21:00:29",
        "2023-01-08 21:03:13",
        "2023-01-08 21:20:29",
        "2023-01-09 23:54:33",
        "2023-01-10 11:22:59",
        "2023-01-11 18:34:49",
        "2023-01-11 18:34:49"
    ])
})

customer_orders


In [0]:
rider_orders = pd.DataFrame({
    "order_id": [1,2,3,4,5,6,7,8,9,10],
    "rider_id": [1,1,1,2,3,3,2,2,2,1],
    "pickup_time": [
        "2023-01-01 18:15:34",
        "2023-01-01 19:10:54",
        "2023-01-03 00:12:37",
        "2023-01-04 13:53:03",
        "2023-01-08 21:10:57",
        None,
        "2023-01-08 21:30:45",
        "2023-01-10 00:15:02",
        None,
        "2023-01-11 18:50:20"
    ],
    "distance": ["5km","6km","4.2km","5.5km","3.3km",None,"6.1km","7.2km",None,"2.8km"],
    "duration": ["32 minutes","27 minutes","20 mins","40","15",None,"25mins","15 minute",None,"10minutes"],
    "cancellation": ["","",None,None,None,"Restaurant Cancellation",None,None,"Customer Cancellation",None]
})

rider_orders["pickup_time"] = pd.to_datetime(rider_orders["pickup_time"])
rider_orders


In [0]:
pizza_names = pd.DataFrame({
    "pizza_id": [1, 2],
    "pizza_name": ["Paneer Tikka", "Veggie Delight"]
})

pizza_names


In [0]:
pizza_recipes = pd.DataFrame({
    "pizza_id": [1, 2],
    "toppings": ["1, 2, 3, 4, 5, 6, 8, 10", "4, 6, 7, 9, 11, 12"]
})

pizza_recipes


In [0]:
pizza_toppings = pd.DataFrame({
    "topping_id": [1,2,3,4,5,6,7,8,9,10,11,12],
    "topping_name": [
        "Paneer","Schezwan Sauce","Tandoori Chicken","Cheese",
        "Corn","Mushrooms","Onions","Capsicum",
        "Red Peppers","Black Olives","Tomatoes","Mint Mayo"
    ]
})

pizza_toppings


### riders (rider_id, registration_date)
### customer_orders (order_id, customer_id, pizza_id, exclusions, extras, order_time)
###  rider_orders (order_id, rider_id, pickup_time, distance, duration, cancellation)
###  pizza_names (pizza_id, pizza_name)
###  pizza_recipes (pizza_id, toppings)
###  pizza_toppings (topping_id, topping_name)
### - ### 

In [0]:
# 1. How many pizzas were ordered?
#first_solution
number_order=len(customer_orders)
number_order
#second solution
n_order=customer_orders.shape[0]
n_order

In [0]:
# 2. How many unique customer orders were made?
unique_order=customer_orders["order_id"].nunique()
unique_order

In [0]:
# 3. How many successful orders were delivered by each rider?
successfull_order=rider_orders[rider_orders["pickup_time"].notna()]
total_order=successfull_order.groupby("rider_id")["pickup_time"].size().reset_index(name="number_of_successful_order")
total_order

In [0]:
customer_rider_delivered = (
    customer_rider.groupby("pizza_id").size().reset_index(name="delivered_count")
    .merge(pizza_names, on="pizza_id", how="left")
)

In [0]:
# 4. How many of each type of pizza was delivered?
customer_rider=customer_orders.merge(rider_orders,on="order_id",how="left")
customer_rider=customer_rider[customer_rider["pickup_time"].notna()]
customer_rider_delivered=(customer_rider.groupby("pizza_id").size().reset_index(name="delivered_count").merge(pizza_names, on="pizza_id", how="left"))
customer_rider_delivered

In [0]:
# 5. How many 'Paneer Tikka' and 'Veggie Delight' pizzas were ordered by each customer?
customer_pizza=customer_orders.merge(pizza_names,on="pizza_id",how="left")
customer_pizza=customer_pizza.groupby(["customer_id","pizza_id","pizza_name"]).size().reset_index(name="number_of_pizza")
customer_pizza


In [0]:
# 6. What was the maximum number of pizzas delivered in a single order?
customer_rider_count=customer_rider.groupby("order_id")["pickup_time"].size().reset_index(name="maximum_number").sort_values("maximum_number",ascending=False)
maximum=customer_rider_count.max()
maximum

In [0]:
c_r_c=customer_rider.groupby("order_id").size().max()
c_r_c

In [0]:
# 7. For each customer, how many delivered pizzas had at least 1 change (extras or exclusions) and how many had no changes?
import numpy as np 
customer_orders_changes=customer_orders.groupby("customer_id")[["extras","exclusions"]].count()
customer_orders_changes["total"]=customer_orders_changes["extras"]+customer_orders_changes["exclusions"]
customer_orders_changess=customer_orders_changes[customer_orders_changes["total"]>1]
customer_orders_changess

In [0]:
customer_rider
customer_rider_delivere=customer_rider[customer_rider["pickup_time"].notna()]
customer_rider_delivere["has_changed"]=np.where((customer_rider_delivere["extras"].notna()) | customer_rider_delivere["exclusions"].notna(),
                                                "has_changed",
                                                "not changed")
result=customer_rider_delivere.groupby(["customer_id", "has_changed"]).size().unstack(fill_value=0).reset_index()
result

In [0]:
# 8. How many pizzas were delivered that had both exclusions and extras?
customer_rider_deliver=customer_rider[customer_rider["pickup_time"].notna()]
both_customer_rider=(customer_rider_deliver["extras"].notna()) & (customer_rider_deliver["exclusions"].notna())
result=both_customer_rider.shape[0]
result

In [0]:
# 9. What was the total volume of pizzas ordered for each hour of the day?
customer_orders["order_time"]=pd.to_datetime(customer_orders["order_time"])
customer_orders["hour"] = customer_orders["order_time"].dt.hour
customer_orders_volume=customer_orders.groupby("hour") \
                        .size() \
                        .reset_index(name="total_volumn")
customer_orders_volume

In [0]:
# 10. What was the volume of orders for each day of the week?
customer_orders["day_name"] = customer_orders["order_time"].dt.day_name()
customer_orders_volumn_week=customer_orders.groupby("day_name").size().reset_index(name="total_volumn")
customer_orders_volumn_week

In [0]:
# 11. How many riders signed up for each 1-week period starting from 2023-01-01?
total_rider=riders.resample("W",on="registration_date").size().reset_index(name="")
total_rider

In [0]:
# 12. What was the average time in minutes it took for each rider to arrive at Pizza Delivery HQ to pick up the order?
customer_orders_rider=customer_orders.merge(rider_orders,on="order_id",how="left")
customer_orders_rider["pickup_time"] = pd.to_datetime(customer_orders_rider["pickup_time"])

customer_orders_rider=customer_orders_rider[customer_orders_rider["pickup_time"].notna()]
customer_orders_rider["du_time"]=(customer_orders_rider["pickup_time"] - customer_orders_rider["order_time"]).dt.total_seconds() / 60

customer_orders_rider_avg=customer_orders_rider.groupby("rider_id")["du_time"].mean().reset_index(name="mean_Time")
customer_orders_rider_avg

In [0]:
# 13. Is there any relationship between the number of pizzas in an order and how long it takes to prepare?
customer_orders_rider
customer_orders_rider_count=customer_orders_rider.groupby("order_id").agg(
    pizza_count=("pizza_id","size"),
    prep_time_minutes=("du_time","mean")
).reset_index()
#correalation option 1
#customer_orders_rider_count[["pizza_count","prep_time_minutes"]].corr()
# option 2
customer_orders_rider_counts=customer_orders_rider_count.groupby("pizza_count")["prep_time_minutes"].mean().reset_index()
customer_orders_rider_counts

In [0]:
# 14. What was the average distance traveled for each customer?
#rstrip()

customer_rider=customer_orders.merge(rider_orders,on="order_id",how="left")
customer_rider=customer_rider[customer_rider["pickup_time"].notna()]
customer_rider["update_distance"]=customer_rider["distance"].str.rstrip("km").astype(float)
avg_distance=customer_rider.groupby("customer_id")["update_distance"].mean().reset_index()
avg_distance

In [0]:
# 14. What was the average distance traveled for each customer?

customer_rider = customer_orders.merge(rider_orders, on="order_id", how="left")
customer_rider = customer_rider[customer_rider["pickup_time"].notna()]
# safe cleaning
customer_rider["distance_km"] = (
    customer_rider["distance"]
    .str.extract(r"([\d\.]+)", expand=False)
    .astype(float)
)
avg_distance = (customer_rider.groupby("customer_id")["distance_km"].mean().reset_index(name="avg_distance_km"))

avg_distance


In [0]:
# 15. What was the difference between the longest and shortest delivery durations across all orders?
rider_orders["update_duration"]=pd.to_numeric(rider_orders["duration"].str[:2], errors="coerce")
rider_orders_diff=(rider_orders["update_duration"].max())-(rider_orders["update_duration"].min())
rider_orders_diff

#rider_orders["duration_minutes"] = (rider_orders["duration"].str.extract(r"(\d+)", expand=False).astype(float))  ##cleaning using regax

In [0]:
# 16. What was the average speed (km/h) for each rider per delivery?
rider_orders["distance_km"]= (rider_orders["distance"].str \
                    .extract(r"([\d\.]+)",expand=False).astype(float)
)
rider_orders["duration_min"] = (rider_orders["duration"].str \
    .extract(r"(\d+)", expand=False).astype(float))
delivered = rider_orders[rider_orders["distance_km"].notna()& rider_orders["duration_min"].notna()]

delivered["speed_kmh"] = (delivered["distance_km"] / (delivered["duration_min"] / 60))

avg_speed = (
    delivered.groupby("rider_id")["speed_kmh"].mean().reset_index(name="avg_speed_kmh")
)
avg_speed

In [0]:
# 17. What is the successful delivery percentage for each rider?
#(success / total) * 100
total_order_per_rider=rider_orders.groupby("rider_id").size().reset_index(name="total_order_per_rider")
successful_orders=rider_orders[rider_orders["pickup_time"].notna()].groupby("rider_id").size().reset_index(name="successful_orders")
deliver_percentage=total_order_per_rider.merge(successful_orders,on="rider_id",how="left")
deliver_percentage["successful_orders"] = ( deliver_percentage["successful_orders"].fillna(0))

deliver_percentage["success_percentage"] = (deliver_percentage["successful_orders"]/ deliver_percentage["total_order_per_rider"] * 100)
deliver_percentage

In [0]:
# 18. What are the standard ingredients for each pizza?
#pizza_name,pizza_recepie,pizza_toppings,  df["items"].str.split(",").explode()
pizza_names_recepie = pizza_names.merge(pizza_recipes, on="pizza_id", how="left")
pizza_names_recepie["toppings"] = pizza_names_recepie["toppings"].str.split(",")
pizza_names_recepie = pizza_names_recepie.explode("toppings").reset_index(drop=True)
pizza_names_recepie["toppings"] = pizza_names_recepie["toppings"].str.strip().astype(int)
pizza_names_recepie_topping = pizza_names_recepie.merge(pizza_toppings, left_on="toppings", right_on="topping_id", how="left")
pizza_names_recepie_topping[["pizza_id","pizza_name","toppings","topping_name"]]

In [0]:
# 19. What was the most commonly added extra (e.g., Mint Mayo, Corn)?
customer_orders_update = customer_orders[customer_orders["extras"].notna()].copy()
customer_orders_update["extras"] = customer_orders_update["extras"].str.split(",")
customer_orders_update = customer_orders_update.explode("extras").reset_index(drop=True)
customer_orders_update["extras"] = customer_orders_update["extras"].str.strip()
customer_orders_update = customer_orders_update[customer_orders_update["extras"].str.isnumeric()]
customer_orders_update["extras"] = customer_orders_update["extras"].astype("int")
extras_with_name = customer_orders_update.merge(pizza_toppings, left_on="extras", right_on="topping_id", how="left")
extras_with_name_count=extras_with_name.groupby("topping_name").size().reset_index(name="total")
mostly_added=extras_with_name_count.sort_values("total",ascending=False)
mostly_added.head(1)

In [0]:
customer_orders

In [0]:
# 20. What was the most common exclusion (e.g., Cheese, Onions)?
customer_orders_update_e = customer_orders[customer_orders["exclusions"].notna()].copy()
customer_orders_update_e["exclusions"] = customer_orders_update_e["exclusions"].str.split(",")
customer_orders_update_e = customer_orders_update_e.explode("exclusions").reset_index(drop=True)
customer_orders_update_e["exclusions"] = customer_orders_update_e["exclusions"].str.strip()
customer_orders_update_e = customer_orders_update_e[customer_orders_update_e["exclusions"].str.isnumeric()]
customer_orders_update_e["exclusions"] = customer_orders_update_e["exclusions"].astype("int")
exclusion_with_name = customer_orders_update_e.merge(pizza_toppings, left_on="exclusions", right_on="topping_id", how="left")
exclusion_with_name_count=exclusion_with_name.groupby("topping_name").size().reset_index(name="total")
mostly_added=exclusion_with_name_count.sort_values("total",ascending=False)
mostly_added.head(1)

In [0]:
# 21. Generate an order item for each record in the `customer_orders` table in the format:

#     * Paneer Tikka
#     * Paneer Tikka - Exclude Corn
#     * Paneer Tikka - Extra Cheese
#     * Veggie Delight - Exclude Onions, Cheese - Extra Corn, Mushrooms
customer_orders_with_pizzaname = customer_orders.merge(pizza_names,on="pizza_id",how="left")
customer_orders_with_pizzaname["extras"] = customer_orders_with_pizzaname["extras"].fillna("0")
customer_orders_with_pizzaname["extras"] = customer_orders_with_pizzaname["extras"].str.split(",")
customer_orders_with_pizzaname = customer_orders_with_pizzaname.explode("extras").reset_index(drop=True)
customer_orders_with_pizzaname["extras"] = customer_orders_with_pizzaname["extras"].str.strip()
customer_orders_with_pizzaname = customer_orders_with_pizzaname[customer_orders_with_pizzaname["extras"].str.isnumeric()]
customer_orders_with_pizzaname["extras"] = customer_orders_with_pizzaname["extras"].astype("int")
extras_with_name = customer_orders_with_pizzaname.merge(pizza_toppings, left_on="extras", right_on="topping_id", how="left")
extras_with_name_count = extras_with_name.groupby("topping_name").size().reset_index(name="total")
mostly_added = extras_with_name_count.sort_values("total", ascending=False)
customer_orders_with_pizzaname
customer_orders_with_pizzaname["exclusions"] = customer_orders_with_pizzaname["exclusions"].fillna("0")
customer_orders_with_pizzaname["exclusions"] = customer_orders_with_pizzaname["exclusions"].str.split(",")
customer_orders_with_pizzaname = customer_orders_with_pizzaname.explode("exclusions").reset_index(drop=True)
customer_orders_with_pizzaname["exclusions"] = customer_orders_with_pizzaname["exclusions"].str.strip()
customer_orders_with_pizzaname = customer_orders_with_pizzaname[customer_orders_with_pizzaname["exclusions"].str.isnumeric()]
customer_orders_with_pizzaname["exclusions"] = customer_orders_with_pizzaname["exclusions"].astype("int")
exclusions_with_name = customer_orders_with_pizzaname.merge(pizza_toppings, left_on="exclusions", right_on="topping_id", how="left")
exclusions_with_name_count = extras_with_name.groupby("topping_name").size().reset_index(name="total")
mostly_added = exclusions_with_name_count.sort_values("total", ascending=False)
customer_orders_with_pizzaname

In [0]:
# 22. Generate an alphabetically ordered, comma-separated ingredient list for each pizza order, using "2x" for duplicates.

#     * Example: "Paneer Tikka: 2xCheese, Corn, Mushrooms, Schezwan Sauce"


In [0]:
# 23. What is the total quantity of each topping used in all successfully delivered pizzas, sorted by most used first?
#successfully delivered pizzas,each topping,total quantity of each topping ,sorted by most used first
successfully_delivered=rider_orders[rider_orders["pickup_time"].notna()]
successfully_delivered_pizza=successfully_delivered.merge(customer_orders,on="order_id",how="left")
successfully_delivered_pizza=successfully_delivered_pizza.merge(pizza_recipes,on="pizza_id",how="left")
successfully_delivered_pizza["toppings"] = successfully_delivered_pizza["toppings"].str.split(",")
successfully_delivered_pizza = successfully_delivered_pizza.explode("toppings")
successfully_delivered_pizza["toppings"] = (successfully_delivered_pizza["toppings"].str.strip())
topping_usage = (successfully_delivered_pizza["toppings"].value_counts().reset_index())
topping_usage.columns = ["topping", "total_quantity"]
topping_usage

In [0]:

# 24. If a 'Paneer Tikka' pizza costs ₹300 and a 'Veggie Delight' costs ₹250 (no extra charges), how much revenue has Pizza Delivery India generated (excluding cancellations)?
import numpy as np
customer_orders["cost"]=(np.where(
    customer_orders["pizza_id"]==1,
    300,
    250
))
customer_orders_rider=customer_orders.merge(rider_orders,on="order_id",how="left")
customer_orders_rider
rider_orders_update=customer_orders_rider[(customer_orders_rider["cancellation"].isna())]
rider_orders_update=rider_orders_update["cost"].sum()
rider_orders_update

In [0]:
price_map = {1: 300, 2: 250}
customer_orders["cost"] = customer_orders["pizza_id"].map(price_map)

customer_orders_rider = customer_orders.merge(
    rider_orders, on="order_id", how="left"
)

successful_orders = customer_orders_rider[
    customer_orders_rider["cancellation"].isna()
]

total_revenue = successful_orders["cost"].sum()
total_revenue


In [0]:
# 25. What if there’s an additional ₹20 charge for each extra topping?
customer_orders["cost"]=(np.where(
    customer_orders["pizza_id"]==1,
    300,
    250
))
customer_orders["extras"]=customer_orders["extras"].str.split(",")
customer_orders = customer_orders.explode("extras").reset_index(drop=True)
customer_orders["extras"] = customer_orders["extras"].str.strip()
customer_orders_u=customer_orders.groupby("order_id").size().reset_index(name="extra_count")

# customer_orders["updated_cost"]=np.where(customer_orders["extras"].notna() & (customer_orders["extras"] != ""),
#                                  customer_orders["cost"]+20,
#                                  customer_orders["cost"]
# )
                                 
customer_orders_u

In [0]:
# 26. Cheese costs ₹20 extra — apply this specifically where Cheese is added as an extra.


In [0]:

# 27. Design a new table for customer ratings of riders. Include:

#     * rating_id, order_id, customer_id, rider_id, rating (1-5), comments (optional), rated_on (DATETIME)

#     Example schema:

#     ```sql
#     CREATE TABLE pizza_delivery_india.rider_ratings (
#       rating_id INT IDENTITY PRIMARY KEY,
#       order_id INT,
#       customer_id INT,
#       rider_id INT,
#       rating INT CHECK (rating BETWEEN 1 AND 5),
#       comments NVARCHAR(255),
#       rated_on DATETIME
#     );
#     ```

In [0]:
# 28. Insert sample data into the ratings table for each successful delivery.


In [0]:
# 29. Join data to show the following info for successful deliveries:

#     * customer_id
#     * order_id
#     * rider_id
#     * rating
#     * order_time
#     * pickup_time
#     * Time difference between order and pickup (in minutes)
#     * Delivery duration
#     * Average speed (km/h)
#     * Number of pizzas in the order


In [0]:

# 30. If Paneer Tikka is ₹300, Veggie Delight ₹250, and each rider is paid ₹2.50/km, what is Pizza Delivery India's profit after paying riders?